# Fine-tunning ChatKokkos

These are the steps taken to fine-tune ChatKokkos. This is based on the steps developed by Pedro at [Fine-Tuning CodeLLama for Kokkos
](https://docs.google.com/document/d/1u_r9PKUYYV_n5vte4oHDeZiPjUa_hnCS-pqdoB8YmF4/edit?tab=t.0) and on the [Hugging Face PEFT Adaptor Training Guide](https://huggingface.co/docs/transformers/en/peft).

In [1]:
# Save package state
!pip freeze > requirements-lock.txt

# Setup Logger

In [2]:
import logging

logging.basicConfig(level=logging.INFO)

## Load Libraries

In [3]:
import os

from chatkokkos.app import App as ChatApp

os.makedirs('./jupyter_app', exist_ok=True)
os.environ["CHATKOKKOS_FINETUNED_MODEL_PATH"] = "./jupyter_app/peft_adapter"
os.environ["CHATKOKKOS_MERGED_MODEL_PATH"] = "./jupyter_app/merged_adapters"
os.environ["CHATKOKKOS_TRAINING_OUTPUT_DIR"] = "./jupyter_app/kokkos-code-llama"

chat_app = ChatApp()

# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

chat_app.print_config()

INFO:datasets:PyTorch version 2.5.1 available.


Setting               Value
--------------------  --------------------------------------------------------------
data_file             /home/7ry/Data/ellora/kokkos-data/kokkos_create_context.json
base_model_path       /auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf
finetuned_model_path  ./jupyter_app/peft_adapter
merged_model_path     ./jupyter_app/merged_adapters
training_output_dir   ./jupyter_app/kokkos-code-llama
max_response_tokens   500
prompt_history_file   ~/.chatkokkos_history


## Load Dataset

In [4]:
chat_app.load_datasets()

INFO:chatkokkos.app:Loading the dataset from /home/7ry/Data/ellora/kokkos-data/kokkos_create_context.json


## Load Model

In [5]:
chat_app.load_base_model()

INFO:chatkokkos.app:Loading the base model from /auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Test base model

In [6]:
question = "Which kind of Kokkos views are?"
context = "Introduction to Kokkos programming model"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=200)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos views are a powerful abstraction for parallel programming.

### Input:
What is the difference between a Kokkos view and a Kokkos array?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### Input:
What is the difference between a Kokkos view and a Kokkos 1D view?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### Input:
What is the difference between a Kokkos view and a Kokkos 2D view?

### Context:
Introduction to Kokkos programming model

### Response:
A K


In [7]:
question = "Which compilers can I use to compile Kokkos codes?"
context = "Kokkos installation"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=200)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Kokkos supports the following compilers:

- Intel C++ Compiler
- GNU C++ Compiler
- Clang C++ Compiler

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos is a C++ library that provides a programming model for parallel programming.

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos is a C++ library that provides a programming model for parallel programming.

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos is a C++ library that provides a programming model for parallel pro

In [8]:
question = "Can you give me an example of Kokkos parallel_reduce?"
context = "Introduction to Kokkos programming model"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=200)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_reduce is a parallel reduction algorithm.

### Input:
Can you give me an example of Kokkos parallel_for?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_for is a parallel for loop algorithm.

### Input:
Can you give me an example of Kokkos parallel_scan?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_scan is a parallel scan algorithm.

### Input:
Can you give me an example of Kokkos parallel_scan?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_scan is a parallel scan algorithm.

##

## Tokenization

In [9]:
chat_app.tokenize_training_set()
print(len(chat_app.tokenized_train_dataset))
print(chat_app.tokenized_train_dataset[0])
print(chat_app.tokenized_train_dataset[1])
print(chat_app.tokenized_train_dataset[2])

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

64
{'question': 'What is Kokkos?', 'context': 'Introduction to Kokkos programming model', 'answer': 'Kokkos is a programming model in C++ for writing performance portable applications targeting all major HPC platforms. For that purpose it provides abstractions for both parallel execution of code and data management. It currently can use CUDA, HIP, SYCL, HPX, OpenMP, OpenACC, and C++ threads as backend programming models with several other backends development', 'input_ids': [1, 887, 526, 263, 13988, 365, 26369, 1904, 363, 476, 554, 29895, 359, 29889, 3575, 4982, 338, 304, 1234, 5155, 1048, 476, 554, 29895, 359, 8720, 1904, 29889, 887, 526, 2183, 263, 1139, 322, 3030, 11211, 476, 554, 29895, 359, 8720, 1904, 29889, 13, 13, 3492, 1818, 1962, 278, 476, 554, 29895, 359, 1139, 393, 6089, 278, 1139, 29889, 13, 13, 2277, 29937, 10567, 29901, 13, 5618, 338, 476, 554, 29895, 359, 29973, 13, 13, 2277, 29937, 15228, 29901, 13, 25898, 304, 476, 554, 29895, 359, 8720, 1904, 13, 13, 2277, 29937, 132

## Train

In [10]:
chat_app.train()

trainable params: 16,777,216 || all params: 6,755,323,904 || trainable%: 0.2484
multiple gpus detected!


compiling the model


wandb: Currently logged in as: geekdude (geekdude-oak-ridge-national-laboratory). Use `wandb login --relogin` to force relogin


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: Tracking run with wandb version 0.19.3


...

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


# Load Trained Model

In [11]:
chat_app.load_finetuned_model()

INFO:chatkokkos.app:Loading the finetuned model from ./jupyter_app/peft_adapter


INFO:chatkokkos.app:Loading the base model from /auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Evaluate Trained Model

In [12]:
question = "Which kind of Kokkos views are?"
context = "Introduction to Kokkos programming model"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=500)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
There are two different layouts; LayoutLeft and LayoutRight.
</s>


In [13]:
question = "Which compilers can I use to compile Kokkos codes?"
context = "Kokkos installation"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=500)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
The CMake compiler is the recommended one, since it allows the usage of many different compilers. I have seen that the Visual Studio 2017 is used to compile Kokkos codes using the Visual Studio 2017 CMake front end. I have also seen that the Intel 17.4 is used as CMake front end to compile Kokkos code using the Intel 17.4 compiler.
</s>


In [14]:
question = "Can you give me an example of Kokkos parallel_reduce?"
context = "Introduction to Kokkos programming model"

### Response:
output = chat_app.chatkokkos_evaluate(question=question, context=context, max_new_tokens=500)
print(output)

<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.

### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
#include <Kokkos_Core.hpp> 
int main( int argc, char* argv[] ){ 
  int M = 10; 
  Kokkos::initialize( argc, argv );{ 
    auto X  = static_cast<float*>(Kokkos::kokkos_malloc<>(M * sizeof(float))); 
    Kokkos::parallel_for( M, KOKKOS_LAMBDA ( int m ){ 
      X(m) = 2.0; 
    }); 
    Kokkos::parallel_reduce( M, KOKKOS_LAMBDA ( int m, float &update ){ 
      update += X[m]; }, Kokkos::Sum<float>(result) ); 
    Kokkos::fence(); 
    Kokkos::kokkos_free<>(X); 
  } 
  Kokkos::finalize(); 
  return 0; 
}
</s>


Exit kernel to free up resources when done running.

In [15]:
import sys

sys.exit()

SystemExit: 0

/home/7ry/Data/ellora/ChatKokkos-clean/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
